In [ ]:
#other imports
import os
import numpy as np
import scipy as sp
import pickle
os.chdir('C:/Code/Github/GLM-analysis/')

In [ ]:
# First setup paths
from utils.path_utils import setup_paths
# from utils.initialize import init_modules
base_dir = setup_paths()

# Now import helper_functions
from helper_functions.data_trial_plotter import TrialPlotter
from helper_functions.data_trial_divider import TrialDivider

# Import core modules
from handlers.DataHandlerDecoding import DataHandlerDecoding as datafun
from utils.Plotter import Plotter as plotterfun
from analysis.DecoderAnalyzer import DecoderAnalyzer as analysisdec
from analysis.AnalysisManagerEncoding import AnalysisManagerEncoding as analysisenc
# Import config after handlers
from config.DatasetConfig import DatasetConfig

# Import local utilities
from utils.dataset_processor import DatasetProcessor
from utils.cell_visualizer import CellVisualizer
from utils.general_stats import GeneralStats

In [ ]:
#initialize class
# decoded_variables= {'sound_category', 'choice', 'photostim', 'outcome','shuffled/sound_category', 'shuffled/choice', 'shuffled/photostim', 'shuffled/outcome'}
# decoded_variables= {'sound_category','shuffled/sound_category'}#{'choice', 'shuffled/choice'}#{'sound_category','shuffled/sound_category', 'choice', 'shuffled/choice','outcome','shuffled/outcome'}
decoded_variables = {'sound_category','shuffled/sound_category','choice', 'shuffled/choice','outcome','shuffled/outcome'}#{'outcome','shuffled/outcome'} #{'sound_category','shuffled/sound_category'}# #{'photostim','shuffled/photostim'} #,'photostim','shuffled/photostim'}
# decoded_variables = {'sound_category','shuffled/sound_category','choice', 'shuffled/choice'}
# decoded_variables = {'outcome','shuffled/outcome'}
                    
data_handler = datafun(decoded_variables=decoded_variables)  # Pass any initial data if needed
data_handler_pass = datafun(decoded_variables=decoded_variables)  # Pass any initial data if needed

In [ ]:
config = DatasetConfig()

info_dir = 'V:/Connie/results/opto_2024/context/mod'
all_datasets, mouse_dates_keys = config.load_from_info(info_dir, data_handler)
datasets, filtered_keys = config.get_datasets_with_variables(decoded_variables,require_all = False, include_datasets = range(25)) #require_all=True loads minimum variables present in all datasets  range(25)

# datasets, filtered_keys = config.get_specific_datasets([0,1])
# event onsets: [7,39,71,132,146]


In [ ]:
config = DatasetConfig()

info_dir = 'V:/Connie/results/opto_2024/context/mod'
all_datasets, mouse_dates_keys = config.load_from_info(info_dir, data_handler)
datasets, filtered_keys = config.get_datasets_with_variables(decoded_variables,include_datasets = range(25)) #range(25)
# datasets2, filtered_keys2 = config.get_datasets_with_variables(decoded_variables,include_datasets = [0,1,4,10,12,13,15,17,24]) #require_all=True loads minimum variables present in all datasets
# results_pre_b,results_pre_all_b, _, celltype_info2 = data_handler.process_multiple_datasets(datasets,'GLM_3nmf_pre',single_balanced=True) 

In [ ]:
#LOAD ACTUAL DATASETS 
save_results = os.path.join(f'W:\Connie/results\Bassi2025/fig2/glm_decoding/') # os.path.join(f'V:/Connie/results/glm_decoding/2025/figures/')
os.makedirs(save_results, exist_ok=True)
cat_results = []
cat_results_pass = []

model_type = 'GLM_3nmf_pre'
results_pre,results_pre_all_sb, _, celltype_info = data_handler.process_multiple_datasets(datasets, model_type,single_balanced=True) 
# results_pre_o,results_pre_all_sb_o, _, celltype_info_o = data_handler.process_multiple_datasets(datasets, model_type,single_balanced=True) 
# ## results_pre_all['HA11-1R_2023-04-13']['sound_category'].keys() - gives means across sc or pop
# if 'photostim' in decoded_variables or 'sound_category' in decoded_variables:
#     model_type = 'GLM_3nmf_passive'
#     results_pass,results_pass_all_sb, cat_results_pass, celltype_info = data_handler_pass.process_multiple_datasets(datasets, model_type,single_balanced=True) 

In [ ]:
decoded_variables_2 ={'outcome','shuffled/outcome'}
                    
data_handler_2 = datafun(decoded_variables=decoded_variables_2)  # Pass any initial data if needed

config_2 = DatasetConfig()

info_dir = 'V:/Connie/results/opto_2024/context/mod'
all_datasets, mouse_dates_keys = config_2.load_from_info(info_dir, data_handler)
datasets, filtered_keys = config_2.get_datasets_with_variables(decoded_variables_2,require_all = False, include_datasets = range(25)) #require_all=True loads minimum variables present in all datasets  range(25)

model_type = 'GLM_3nmf_pre'
results_pre_outcome,results_pre_all_sb_outcome, _, _ = data_handler_2.process_multiple_datasets(datasets, model_type,single_balanced=True) 


In [ ]:
#Initialize plotting class
save_results = os.path.join(f'W:\Connie/results\Bassi2025/fig2/glm_decoding/')
plotter = plotterfun(data = None, save_results= save_results)
#initialize analysis class from encoding bc we are using the same functions
analysisenc = analysisenc(data = None, plotter = plotter)

In [ ]:
# Get shuffled data into frames x neurons x total shuffles across splits (50 * 10 = 500)
print(list(decoded_variables)[0])
print(list(decoded_variables))
shuffled_structure = data_handler.create_shuffled_distribution_structure(decoder_type = list(decoded_variables)[2], metric = 'sc_instantaneous_information')
# if 'photostim' in decoded_variables or 'sound_category' in decoded_variables:
#     shuffled_structure_pass = data_handler_pass.create_shuffled_distribution_structure(decoder_type = list(decoded_variables)[1], metric = 'sc_instantaneous_information')
  

In [ ]:
# Plot population decoders across datasets
current_model_type = 'pre'
results_pre_all = results_pre_all_sb
current_decoder_type = list(decoded_variables)[0] #'choice'   
os.makedirs(f'{plotter.save_results}/{current_model_type}/', exist_ok=True)

# Usage example:
plotter.plot_decoding_results(results_pre_all, 
                     decoder_type= current_decoder_type,
                     plot_type='pop', # or sc
                     save_dir= None, #f'{plotter.save_results}/{current_model_type}', #,save_dir = f'{save_results}/{current_decoder_type}_{current_model_type}_pop'
                     xlim = (0,169),
                     ylim = [0.7, 0.7, .9, .9]) #define y limits for each subplot

In [ ]:
# List of mean_results and corresponding decoder types
mean_results_list = [results_pre_all_sb,results_pre_all_sb]#[results_pre_all_sb, results_pass_all_sb]
current_decoded_variable = 'outcome' # 'choice' or 'sound_category' or 'photostim' or 'outcome'
decoder_types = [current_decoded_variable,f'shuffled/{current_decoded_variable}'] #['sound_category','sound_category'] #['sound_category','sound_category'] #['choice','shuffled/choice'] ##['sound_category','sound_category'] #['photostim','photostim'] #['outcome', 'shuffled/outcome']

# Plot population decoders across datasets
current_model_type = 'both'#'pre' or passive
data_type = 'fraction_correct' # 'fraction_correct' or 'information'
info_type = 'instantaneous' #'cumulative' #'instantaneous'

if data_type == 'information':
    ylabel = 'Bits'
    ylimss = (0.0,.2)
    if info_type == 'cumulative':
        ylimss = (0.0,.5)
else:
    ylabel = '% Accuracy'
    ylimss = (0.45,.8)
    if info_type == 'cumulative':
        ylimss = (0.45,.8)
current_decoder_type = decoder_types[0]
save_dir_contexts = f'{plotter.save_results}' # #f'{plotter.save_results}/{current_model_type}_{info_type}_{data_type}_{current_decoder_type}' 

# Determine labels based on mean_results_list
try:
    # First check if results_pass_all_sb exists and is part of mean_results_list
    if 'results_pass_all_sb' in locals() and mean_results_list == [results_pre_all_sb, results_pass_all_sb]:
        labels = ['Active', 'Passive']
        decoder_types = [current_decoded_variable,current_decoded_variable]
    else:
        labels = [current_decoded_variable, f'shuff {current_decoded_variable}']
        labels = ['sound', f'shuff sound']
except NameError:
    # If results_pass_all_sb doesn't exist, use decoded variable labels
    labels = [current_decoded_variable, f'shuff {current_decoded_variable}']
    


colors_used = plotter.default_variable_colors[current_decoded_variable] #['current_decoded_variable']#['mediumvioletred','hotpink']#['saddlebrown','darkorange'] #['steelblue','lightskyblue'] #colors for the bar plot     
frames_used = np.arange(6,35) #frames to plot in the bar/scatter plot (7,22)
os.makedirs(save_dir_contexts , exist_ok=True)


# Plot the selected metric and get concatenated means and SEMs
concatenated_means, concatenated_sems = plotter.plot_selected_metric_with_sem(mean_results_list, decoder_types, 
                                                                      metric=f'pop_{info_type}_{data_type}_mean', 
                                                                      title= None, 
                                                                      xlabel='Seconds', 
                                                                      ylabel= ylabel, #'SC Metric 1 Value',
                                                                      ylim = ylimss, #(0.01,.03)
                                                                      xlim = (0,169), #in terms of frames (30 frames = 1 second)
                                                                    save_dir=save_dir_contexts,
                                                                    labels = labels,
                                                                    colors = colors_used)


# create bar or scatter plot of mean values at specific range of values
plotter.simple_plot_wrapper(labels, concatenated_means, concatenated_sems,plot_type = 'scatter', colors= colors_used, frames = frames_used, ylabel = 'Mean Value', save_dir = save_dir_contexts)


In [ ]:
# Plot individual datasets the selected metric and get concatenated means and SEMs

os.makedirs(f'V:/Connie/results/glm_decoding/{current_decoded_variable}_decoding', exist_ok=True)
for dataset_key in results_pre_all_sb:
  # dataset_key = list(results_pre_all_sb.keys())[0]
  plotter.plot_dataset_metric_with_sem(mean_results_list,dataset_key, decoder_types,  
                            title= None, 
                            metric = f'pop_instantaneous_information_mean',
                            start_frame=0,
                            xlabel='Seconds', 
                            ylabel= ylabel, #'SC Metric 1 Value',
                            ylim = (0.0,.5), #ylimss, #(0.01,.03)
                            xlim = (0,169), #in terms of frames (30 frames = 1 second)
                          save_dir='V:\Connie/results\glm_decoding\sound_decoding', #,#save_dir_contexts,
                          labels = labels,
                          colors = colors_used)       

In [ ]:
# Create an instance of AnalysisManager, passing in the plotter module
analysis_dec = analysisdec(celltype_info)

In [ ]:
#get peaks without thresholding TO GET STATS ACROSS ALL NEURONS
current_model_type = 'pre' #'pre' or passive
results_pre_all = results_pre_all_sb #results_to_test #results_pre_all_sb
start_frame = 14 #14 before this there is artifact
threshold = -.1 #shuffled_threshold['universal_threshold'] #np.max(list(shuffled_threshold['thresholds'].values())) #shuffled_threshold['universal_threshold'] #0.06 #shuffled_threshold['universal_threshold'] #use threshold OR percentile to decide signficant neurons
method = 'threshold_peak'#'shuffled_timepoint' #'threshold_peak'# 'shuffled_peak', 'threshold_peak', 'range_threshold', 'combined', 'shuffled_timepoint'
metric_to_plot = 'sc_instantaneous_information_mean'
current_decoder_type = 'choice' #shuffled/
prtile = 90 

save_path = f'{plotter.save_results}/{current_model_type}{current_decoder_type}/'
os.makedirs(save_path , exist_ok=True)

# # Analyze peaks by cell type
# peaks_by_celltype = []
# peaks_by_celltype = analysis_dec.analyze_peaks_by_celltype( results_pre_all,shuffled_structure, method = method,
#                                                             decoder_type=current_decoder_type, start_frame=start_frame , end_frame = end_frame,
#                                                             significance_percentile = prtile, threshold = threshold, window = 3)


types = 'violin'  # 'cdf' or 'histogram' or 'violin'
decoder_type = current_decoder_type   # 'choice' or 'sound_category' or 'photostim' or 'outcome'

save_path = f'{plotter.save_results}/{decoder_type}/all_cells/'#f'{plotter.save_results}_14to169/{current_model_type}{decoder_type}/' #f'{plotter.save_results}/{current_model_type}{start_frame}{current_decoder_type}/'
os.makedirs(save_path, exist_ok=True)
end_frame = 169
significant_neurons_data, significance_struc, significant_neurons = analysis_dec.analyze_significant_neurons(
            results_dict=results_pre_all_sb,
            shuffled_structure = shuffled_structure,
            method = 'threshold_peak', #threshold_peak', 'range_threshold'
            decoder_type = decoder_type,  # Or your preferred decoder type
            start_frame = 14,                 # Specify start frame
            end_frame = end_frame,  #100                  # Specify end frame if needed, else None
            metric = 'sc_instantaneous_information_mean', #choose sc metric to look at
            significance_percentile = 95,         #used to determine significant cells
            threshold = threshold                    #None or 0.06 (from Runyan 2017)

        )




In [ ]:
## MAKE PLOTS RELATING PEAK DECODING INFO ACROSS DIFFERENT FEATURES (SOUND VS CHOICE, SOUND VS PHOTOSTIM, SOUND VS OUTCOME)
### COMPUTE AND PLOT PEAK INFO BY CELL TYPE
feature1 = 'sound_category'  #'choice' #'sound_category' #'photostim' #'outcome'
feature2 = 'choice'

save_path = f'W:\Connie/results\Bassi2025/fig2\glm_decoding/overlaps/'
if feature1 == 'sound_category':
    frames_to_use = 100
else:   
    frames_to_use = 169
# Compute peak info by cell type
celltype_label_map = {
    0: "Pyr",   # adjust if your convention is different
    1: "SOM",
    2: "PV"
}
import matplotlib.colors as mcolors
#colors
c1 = np.array(mcolors.to_rgb(plotter.default_variable_colors[feature1][0]))
c2 = np.array(mcolors.to_rgb(plotter.default_variable_colors[feature2][0]))

mixed_color = tuple((c1 + c2) / 2)
peak_info_struc, peak_info_all_celltypes = analysisdec.compute_peak_info_by_celltype(results_dict = results_pre_all_sb,
                                  celltype_assignments = {
                                                            ds: results_pre_all_sb[ds]['celltype_array']
                                                            for ds in results_pre_all_sb
                                                        },
                                  features = [feature1,feature2],
                                  metric = 'sc_instantaneous_information_mean',
                                  frame_windows = {
                                                        feature1: (14, frames_to_use),  # adjust as needed
                                                        feature2: (14, 169)          # adjust as needed
                                                    },
                                  celltype_label_map = celltype_label_map)

counts_pyr = analysisdec.quadrant_counts_for_celltype(
    peak_info_struc,
    celltype="Pyr",
    feature_names=(feature1, feature2),
    threshold=0.06
)

plotter.plot_scatter_all_datasets_by_celltype(
    peak_info_struc,
    celltypes=["Pyr","PV","SOM"],
    stim_feature=feature1,
    choice_feature=feature2,
    threshold=0.06,
    figsize= (1.5,1.5),#,(1.3,1.3),#(1.5*2.5,1.5),
    lims = (0,.4), save_path = f'{save_path}/{feature1}_vs_{feature2}_scatter_all_datasets_by_celltype.pdf',
    uniformative_colors = (0.8,0.8,0.8)
)


synergy_struc = analysisdec.compute_synergy_by_celltype(
    peak_info_struc,
    celltypes=["Pyr","SOM","PV"],
    stim_feature=feature1,
    choice_feature=feature2
)
synergy_all = analysisdec.aggregate_synergy_across_datasets(synergy_struc)

plotter.plot_synergy_violin(synergy_all, celltypes=["Pyr","SOM","PV"],colors = ((0.37, 0.75, 0.49),(0.17, 0.35, 0.8),(0.82, 0.04, 0.04)), figsize=(1.5,1.5))

pooled_scatter = analysisdec.synergy_scatter_all(peak_info_struc,feature_names=(feature1,feature2))
plotter.default_variable_colors = {}
plotter.default_variable_colors = {
            'sound_category': ['darkslateblue', 'mediumslateblue'],
            'choice': ['dodgerblue','blue'],#['steelblue', 'lightskyblue'],
            'photostim': ['saddlebrown', 'darkorange'],
            'outcome': ['mediumvioletred', 'hotpink']
        }
plotter.plot_synergy_vs_peak_pooled(pooled_scatter, celltypes=("Pyr","SOM","PV"),figsize = (1.5*3,1.5), threshold = 0.06, xlims = (0,.5), feature_names = (feature1,feature2), 
                            colors = (c1,c2),
                            save_path = f'{save_path}/{feature1}_vs_{feature2}_synergy_vs_peak_pooled.pdf')


# Compute for sound vs choice

info_fracs, venn_f1, venn_f2, venn_both = analysisdec.compute_dataset_fractions(peak_info_struc, feature1, feature2)

plotter.plot_informative_pie(info_fracs, feature1, feature2,figsize = (1.0,1.0),
                     save_path = f'{save_path}/{feature1}_vs_{feature2}_informative_pie.pdf')
plotter.plot_overlap(venn_f1, venn_f2, venn_both, feature1, feature2, mode="venn", figsize = (1.5,1.5),colors =(c1,c2),
             save_path = f'{save_path}/{feature1}_vs_{feature2}_overlap_venn.pdf')

# plotter.plot_overlap_stacked(
#     venn_f1,
#     venn_f2,
#     venn_both,
#     feature1, feature2,
#     figsize=(1.2, 1.5),
#     colors=(plotter.default_variable_colors[feature1][0],plotter.default_variable_colors[feature2][0]),
#     show_error=False,
#     save_path= f'{save_path}/{feature1}_vs_{feature2}_overlap_stackedbars.pdf'
# )

plotter.plot_overlap_pie(
    venn_f1,
    venn_f2,
    venn_both,
    feature1, feature2,
    figsize=(1.2, 1.2),
    colors=(c1,c2,mixed_color),
    save_path= f'{save_path}/{feature1}_vs_{feature2}_overlap_pie.pdf'
)


In [ ]:
#get peaks without thresholding TO GET STATS ACROSS ALL NEURONS
current_model_type = 'pre' #'pre' or passive
results_pre_all = results_pre_all_sb #results_to_test #results_pre_all_sb
start_frame = 14 #14 before this there is artifact
threshold = -.1 #shuffled_threshold['universal_threshold'] #np.max(list(shuffled_threshold['thresholds'].values())) #shuffled_threshold['universal_threshold'] #0.06 #shuffled_threshold['universal_threshold'] #use threshold OR percentile to decide signficant neurons
method = 'threshold_peak'#'shuffled_timepoint' #'threshold_peak'# 'shuffled_peak', 'threshold_peak', 'range_threshold', 'combined', 'shuffled_timepoint'
metric_to_plot = 'sc_instantaneous_information_mean'
current_decoder_type = 'choice' #shuffled/
prtile = 90 

save_path = f'{plotter.save_results}/{current_model_type}{current_decoder_type}/'
os.makedirs(save_path , exist_ok=True)

# # Analyze peaks by cell type
# peaks_by_celltype = []
# peaks_by_celltype = analysis_dec.analyze_peaks_by_celltype( results_pre_all,shuffled_structure, method = method,
#                                                             decoder_type=current_decoder_type, start_frame=start_frame , end_frame = end_frame,
#                                                             significance_percentile = prtile, threshold = threshold, window = 3)


types = 'violin'  # 'cdf' or 'histogram' or 'violin'
decoder_type = current_decoder_type   # 'choice' or 'sound_category' or 'photostim' or 'outcome'

save_path = f'{plotter.save_results}/{decoder_type}/all_cells/'#f'{plotter.save_results}_14to169/{current_model_type}{decoder_type}/' #f'{plotter.save_results}/{current_model_type}{start_frame}{current_decoder_type}/'
os.makedirs(save_path, exist_ok=True)
end_frame = 169
significant_neurons_data, significance_struc, significant_neurons = analysis_dec.analyze_significant_neurons(
            results_dict=results_pre_all_sb,
            shuffled_structure = shuffled_structure,
            method = 'threshold_peak', #threshold_peak', 'range_threshold'
            decoder_type = decoder_type,  # Or your preferred decoder type
            start_frame = 14,                 # Specify start frame
            end_frame = end_frame,  #100                  # Specify end frame if needed, else None
            metric = 'sc_instantaneous_information_mean', #choose sc metric to look at
            significance_percentile = 95,         #used to determine significant cells
            threshold = threshold ,                   #None or 0.06 (from Runyan 2017)

        )

plotter.plot_significant_neurons_distribution(
            significant_neurons_data = significance_struc,
            save_path=f'{save_path}{types}{decoder_type}_{end_frame}sessionmeans_cdf_peak_info.pdf',
            event_frames = plotter.event_frames,
            figure_type=types,
            star_height_percentage = 0.0001,
            figsize = (2.8, 1.6),
            bin_size= 10
        )
# means, sems, percentages_by_celltype = plotter.plot_significant_neuron_percentages_by_celltype( significance_struc,analysis_dec.celltype_info, save_path=f'{save_path}bar_{decoder_type}{end_frame}_percent_informative.pdf', star_height_percentage=-0.05)#-0.1


In [ ]:
#DECIDE WHETHER OR NOT TO USE END FRAME
## shuffled_threshold = analysis_dec.universal_shuffled_threshold(shuffled_structure, start_frame = 14, end_frame = 100, significance_percentile=95)
# shuffled_threshold = analysis_dec.universal_shuffled_threshold(shuffled_structure, start_frame = 14, end_frame = 100, significance_percentile=5, mode='peak')

current_model_type = 'pre' #'pre' or passive
results_pre_all = results_pre_all_sb #results_to_test #results_pre_all_sb
start_frame = 14 #14 before this there is artifact
end_frame = 169 #100 for sound/photostim
threshold = 0.06 #shuffled_threshold['universal_threshold'] #np.max(list(shuffled_threshold['thresholds'].values())) #shuffled_threshold['universal_threshold'] #0.06 #shuffled_threshold['universal_threshold'] #use threshold OR percentile to decide signficant neurons
method = 'threshold_peak'#'shuffled_timepoint' #'threshold_peak'# 'shuffled_peak', 'threshold_peak', 'range_threshold', 'combined', 'shuffled_timepoint'
metric_to_plot = 'sc_instantaneous_information_mean'
current_decoder_type = 'outcome' #shuffled/
prtile = 90 

# Replace dot with underscore in threshold for directory name
# threshold_str = f"{str(threshold).replace('.', '_')}_{method}"#str(threshold).replace('.', '_')
if method in ['threshold_peak', 'range_threshold']:
    threshold_str = str(threshold).replace('.', '_')
else:
    threshold_str = (f'{prtile} _ {method}')

save_path = f'{plotter.save_results}/{current_model_type}{current_decoder_type}/'
os.makedirs(save_path , exist_ok=True)

# Analyze peaks by cell type
peaks_by_celltype = []
peaks_by_celltype = analysis_dec.analyze_peaks_by_celltype( results_pre_all,shuffled_structure, method = method,
                                                            decoder_type=current_decoder_type, start_frame=start_frame , end_frame = end_frame,
                                                            significance_percentile = prtile, threshold = threshold, window = 3) #, end_frame = 100

#PLOT DISTRIBUTIONS OF SIGINIFICANT NEURONS!

cdf_data, cell_labels = analysis_dec.format_peaks_for_cdf(peaks_by_celltype,metric = metric_to_plot, significant_only=True)
all_peaks, neuron_groups = analysis_dec.format_peaks_for_boxplot(peaks_by_celltype, metric = metric_to_plot, significant_only=True)

#cdf
if "shuffled" in current_decoder_type:
    current_decoder_type = current_decoder_type.replace('/', '_') #replace / with _
plotter.plot_cdf_coupling_index(np.array(cdf_data['all']['peaks']),
                                cell_labels,
                                colors = plotter.celltypecolors,
                                title=f'{current_decoder_type}{threshold_str}',
                                xlabel = 'Information (bits)',
                                xval=.15,
                                xint=0.1,
                                save_path=None, #f'{save_path}{current_decoder_type}_cdf_thr{threshold_str}.pdf',
                                perform_stats=True)

#make boxplot!
plotter.box_plot(
    data=all_peaks,
    neuron_groups=neuron_groups,
    colors=plotter.celltypecolors,
    measure_string='Peak Information (bits)',
    save_path=None #f'{save_path}{current_decoder_type}_box_plot_thr{threshold_str}.pdf'
)

#make boxplot!
plotter.bar_plot(
    data=all_peaks,
    neuron_groups=neuron_groups,
    colors=plotter.celltypecolors,
    measure_string='Peak Information (bits)',
    ylims = 0.1,
    save_path=None #f'{save_path}{current_decoder_type}_violin_plot_thr{threshold_str}.pdf'
)

#heatmap across datasets
# if "shuffled" in current_decoder_type:
#     current_decoder_type = current_decoder_type.replace('_', '/') #replace / with _
# plotter.plot_decoding_heatmap_datasets(results_pre_all, decoder_type= current_decoder_type, metric = 'pop_instantaneous_information_mean')

# PRINT OUT # OF SIGNIFICANT NEURONS PER CELL TYPE
# Assuming cdf_data is a dictionary with cell types as keys and peaks as values
for cell_type, data in cdf_data.items():
    if 'peaks' in data:
        num_significant_neurons = len(np.array(data['peaks']))
        print(f'Cell type: {cell_type}, Number of significant neurons: {num_significant_neurons}')


In [ ]:
#save most informative neurons!
import scipy.io

# Sort peaks by information
sorted_peaks,sorted_peaks_more_info = analysis_dec.sort_peaks_by_information(peaks_by_celltype)
# Prepare the dictionary
matlab_ready = analysis_dec.flatten_for_matlab(sorted_peaks) #needed for SVM decoding

# Save to .mat file
scipy.io.savemat(r'V:/Connie/ProcessedData\sorted_peaks\sorted_peaks_outcome_14to169.mat', matlab_ready)

In [ ]:
# Plot significant neurons distribution
types = 'violin'  # 'cdf' or 'histogram' or 'violin'
decoder_type = 'outcome'  # 'choice' or 'sound_category' or 'photostim' or 'outcome'
if decoder_type == 'outcome':
    result_dict_to_use = results_pre_all_sb_outcome
else:
    result_dict_to_use = results_pre_all_sb
    
save_path = f'{plotter.save_results}/{decoder_type}/'#f'{plotter.save_results}_14to169/{current_model_type}{decoder_type}/' #f'{plotter.save_results}/{current_model_type}{start_frame}{current_decoder_type}/'
os.makedirs(save_path, exist_ok=True)

end_frame = 169
if decoder_type == 'sound_category':
    end_frame = 100

significant_neurons_data, significance_struc, significant_neurons = analysis_dec.analyze_significant_neurons(
            results_dict=result_dict_to_use,
            shuffled_structure = shuffled_structure,
            method = 'threshold_peak', #threshold_peak', 'range_threshold'
            decoder_type = decoder_type,  # Or your preferred decoder type
            start_frame = 14,                 # Specify start frame
            end_frame = end_frame,  #100                  # Specify end frame if needed, else None
            metric = 'sc_instantaneous_information_mean', #choose sc metric to look at
            significance_percentile = 95,         #used to determine significant cells
            threshold = 0.06 ,                   #None or 0.06 (from Runyan 2017)

        )



plotter.plot_significant_neurons_distribution(
            significant_neurons_data = significance_struc,
            save_path=f'{save_path}{types}{decoder_type}_{end_frame}sessionmeans_cdf_peak_info.pdf',
            event_frames = plotter.event_frames,
            figure_type=types,
            star_height_percentage = 0.0001,
            figsize = (2.8, 1.4),
            bin_size= 3, plot_peak_locs= False
        )


means, sems, percentages_by_celltype = plotter.plot_significant_neuron_percentages_by_celltype( significance_struc,analysis_dec.celltype_info, save_path=f'{save_path}bar_{decoder_type}{end_frame}_percent_informative.pdf', star_height_percentage=-0.05)#-0.1

print(save_path)

##below plots the means across datasets!
# plotter.plot_significant_neuron_session_means(
#             significant_neurons_data = significance_struc,
#             save_path=f'{save_path}{types}{decoder_type}_{end_frame}sessionmeans_cdf_peak_info.png',
#             event_frames = plotter.event_frames,
#             figure_type=types,
#             star_height_percentage = 0.0001,
#             fig_size = (2.8, 1.6)
#         )

# plotter.plot_significant_neuron_session_means(
#             significant_neurons_data = significance_struc,
#             save_path=f'{save_path}{types}{decoder_type}_{end_frame}sessionmeans_cdf_peak_info.png',
#             event_frames = plotter.event_frames,
#             figure_type=types,
#             star_height_percentage = 0.0001,
#             fig_size = (2.8, 1.6)
#         )

In [ ]:
# Plot significant neurons distribution
types = 'violin'  # 'cdf' or 'histogram' or 'violin'
decoder_type = 'choice'  # 'choice' or 'sound_category' or 'photostim' or 'outcome'

save_path = f'{plotter.save_results}/{decoder_type}/'#f'{plotter.save_results}_14to169/{current_model_type}{decoder_type}/' #f'{plotter.save_results}/{current_model_type}{start_frame}{current_decoder_type}/'
os.makedirs(save_path, exist_ok=True)
end_frame = 169 #169
significant_neurons_data, significance_struc, significant_neurons = analysis_dec.analyze_significant_neurons(
            results_dict=results_pre_all_sb,
            shuffled_structure = shuffled_structure,
            method = 'threshold_peak', #threshold_peak', 'range_threshold'
            decoder_type = decoder_type,  # Or your preferred decoder type
            start_frame = 14,                 # Specify start frame
            end_frame = end_frame,  #100                  # Specify end frame if needed, else None
            metric = 'sc_instantaneous_information_mean', #choose sc metric to look at
            significance_percentile = 95,         #used to determine significant cells
            threshold = 0.06 ,                   #None or 0.06 (from Runyan 2017)

        )

plotter.plot_significant_neurons_distribution(
            significant_neurons_data = significance_struc,
            save_path=f'{save_path}{types}{decoder_type}_{end_frame}sessionmeans_cdf_peak_info.pdf',
            event_frames = plotter.event_frames,
            figure_type=types,
            star_height_percentage = 0.0001,
            figsize = (2.8, 1.6),
            bin_size= 15
        )

plotter.plot_significant_neuron_session_means(
            significant_neurons_data = significance_struc,
            save_path=f'{save_path}{types}{decoder_type}_{end_frame}sessionmeans_cdf_peak_info.pdf',
            event_frames = plotter.event_frames,
            figure_type=types,
            star_height_percentage = 0.0001,
            fig_size = (2.8, 1.6)
        )

plotter.plot_significant_neuron_session_means(
            significant_neurons_data = significance_struc,
            save_path=f'{save_path}{types}{decoder_type}_{end_frame}sessionmeans_cdf_peak_info.pdf',
            event_frames = plotter.event_frames,
            figure_type=types,
            star_height_percentage = 0.0001,
            fig_size = (2.8, 1.6)
        )

means, sems, percentages_by_celltype = plotter.plot_significant_neuron_percentages_by_celltype( significance_struc,analysis_dec.celltype_info, save_path=f'{save_path}bar_{decoder_type}{end_frame}_percent_informative.pdf', star_height_percentage=-0.05)#-0.1

print(save_path)

In [ ]:
plotter.celltypecolors.keys()

In [ ]:
types = 'violin'  # 'cdf' or 'histogram' or 'violin'
decoder_type = 'choice'  # 'choice' or 'sound_category' or 'photostim' or 'outcome'

save_path = f'{plotter.save_results}/{decoder_type}/'#f'{plotter.save_results}_14to169/{current_model_type}{decoder_type}/' #f'{plotter.save_results}/{current_model_type}{start_frame}{current_decoder_type}/'
os.makedirs(save_path, exist_ok=True)
end_frame = 169 #169
significant_neurons_data, significance_struc, significant_neurons = analysis_dec.analyze_significant_neurons(
            results_dict=results_pre_all_sb,
            shuffled_structure = shuffled_structure,
            method = 'threshold_peak', #threshold_peak', 'range_threshold'
            decoder_type = decoder_type,  # Or your preferred decoder type
            start_frame = 14,                 # Specify start frame
            end_frame = end_frame,  #100                  # Specify end frame if needed, else None
            metric = 'sc_instantaneous_information_mean', #choose sc metric to look at
            significance_percentile = 95,         #used to determine significant cells
            threshold = 0.06 ,                   #None or 0.06 (from Runyan 2017)

        )
means, sems, percentages_by_celltype = plotter.plot_significant_neuron_percentages_by_celltype( significance_struc,analysis_dec.celltype_info, save_path=f'{save_path}bar_{decoder_type}{end_frame}_percent_informative.pdf', star_height_percentage=-0.05)


In [ ]:
# MAKE SOME OTHER PLOTS LOOKING AT TEMPORAL DYNAMICS OF DECODING 
# (CELLS BELOW ARE EXPLANATORY ANALYSIS, NOT NECESSARY FOR MAIN FIGURES)
threshold = 0.06
significant_neurons_data, sig_struct, sig_neurons = analysis_dec.wrapper_info_plots_analysis(
    results_dict=results_pre_all_sb,
    shuffled_structure = shuffled_structure,
    method = 'threshold_peak', #threshold_peak', 'range_threshold'
    plotter=plotter,
    decoder_type = 'sound_category',  # Or your preferred decoder type
    start_frame = 14,                 # Specify start frame
    end_frame = 169,  #100                  # Specify end frame if needed, else None
    metric = 'sc_instantaneous_information_mean', #choose sc metric to look at
    significance_percentile = 95,         #used to determine significant cells
    threshold = threshold ,                   #None or 0.06 (from Runyan 2017)
    save_path= None #f'{save_path}{current_decoder_type}_sig_cel_thr{str(threshold)}'
)

# # Save significant neurons data to a .mat file
# mat_file_path = os.path.join(save_path, f'significant_neurons_data_none.mat')
# sp.io.savemat(mat_file_path, significant_neurons_data)


In [ ]:
#LOAD ALIGNED DATA! TO MAKE PLOTS
from helper_functions.data_pipeline import DataPipeline
pipeline = DataPipeline()
data_loaders, celltype_info = pipeline.load_data(datasets=datasets, load_celltypes=True)

# Initialize processor with alignment parameters
processor = DatasetProcessor(alignment={
    'type': 'pre',
    'data_type': 'deconv'
})

fields_to_separate = ['condition','left_turn'] # ['left_turn'] #is_stim_trial, correct, condition
# 
# Create trial divider for condition splitting
trial_divider = TrialDivider()

# Process and align all datasets
aligned_data = processor.process_datasets(
    data_loaders=data_loaders,
    celltype_info=celltype_info,
    save_path='aligned_data.pkl'
)

# For each dataset, create condition splits
for (animalID, date), data in aligned_data.items():
    print(f"\nProcessing {animalID} {date}")
    
    # Create condition splits
    
    all_conditions, condition_array = trial_divider.divide_trials_from_df(
        trial_info_df=data['trial_info'],
        good_trials=data['good_trials'],
        fields_to_separate=fields_to_separate
    )
    
    # Store conditions in aligned_data for later use
    aligned_data[(animalID, date)]['all_conditions'] = all_conditions
    print(f"Found {len(all_conditions)} conditions")

# Verify alignment worked
for (animalID, date), data in aligned_data.items():
    print(f"\nDataset: {animalID} {date}")
    print(f"Aligned imaging shape: {data['aligned_imaging'].shape}")
    print(f"Number of good trials: {len(data['good_trials'])}")
    print(f"Number of conditions: {len(data['all_conditions'])}")

In [ ]:
# Initialize visualization tools
trial_plotter = TrialPlotter()
trial_divider = TrialDivider()
cell_viz = CellVisualizer()

# Create output directories if they don't exist
import os
base_figure_dir = f'{save_results}/figures/{current_decoder_type}'
os.makedirs(f'{base_figure_dir}/heatmaps', exist_ok=True)
os.makedirs(f'{base_figure_dir}/example_cells', exist_ok=True)
print(f"Base figure directory: {base_figure_dir}")

In [ ]:
# PLOT EXAMPLE CELLS (NOT SORTED BY SIGNIFICANCE) - Plot first 3 cells of each type

example_dataset = ['HA11-1R_2023-05-05']
significant_neurons_data = sorted_peaks 
# Plot for each dataset
for dataset in significant_neurons_data:
    # Extract animal ID and date from dataset name
    animalID, date = dataset.split('_')
    
    # Get the aligned data for this dataset
    data = aligned_data[(animalID, date)]
    
    # Create output directory for this dataset
    dataset_dir = os.path.join('figures', 'informative_cells', dataset)
    os.makedirs(dataset_dir, exist_ok=True)
    
    # Plot heatmaps for different cell types
    for cell_type in ['pyr']: #, 'pv', 'som'
        if cell_type in significant_neurons_data[dataset]:
            # Get significant cells for this cell type
            sig_cells = significant_neurons_data[dataset][cell_type]
            
            # Plot example cells for each condition
            for n,cell_id in enumerate(sig_cells[:15]):  # Plot first 3 cells of each type (:3)
                # for trials, comb, label in all_conditions:
                # Create valid filename by replacing problematic characters
                # safe_label = label.replace('/', '_').replace(' ', '_')
                print(f"Plotting {cell_type} cell {cell_id} for {animalID} {date}")
                
                cell_viz.plot_informative_cell(
                    aligned_imaging=data['aligned_imaging'],
                    cell_id=cell_id,
                    all_conditions=data['all_conditions'], #condition_indices=trials,
                    title_base=f'{animalID} {date} {cell_type} Cell {cell_id} ', #- {label}
                    peak_info = sorted_peaks_more_info[dataset][cell_type]['peak_values'][n], #peaks_by_celltype[dataset][cell_type]['sc']['sc_instantaneous_information_mean']['peak_values'][cell_id],
                    save_path=f'{base_figure_dir}/example_cells/{animalID}_{date}_{cell_type}_cell_{cell_id}' #_{safe_label}
                )
print("Plotting complete!")

In [ ]:
example_dataset = ['HA11-1R_2023-05-05']
condition_colors = ['black','gray','black','gray'] #['#377eb8', '#e41a1c']  # blue and red

# Plot for each dataset
for dataset in example_dataset:
    # Extract animal ID and date from dataset name
    animalID, date = dataset.split('_')
    
    # Get the aligned data for this dataset
    data = aligned_data[(animalID, date)]
    
    # Create output directory for this dataset
    dataset_dir = os.path.join('figures', 'informative_cells', dataset)
    os.makedirs(dataset_dir, exist_ok=True)
    
    # Plot heatmaps for different cell types
    for cell_type in ['pyr', 'pv', 'som']:
        if cell_type in significant_neurons_data[dataset]:
            # Get significant cells for this cell type
            sig_cells = significant_neurons_data[dataset][cell_type]
            
            # Plot example cells for each condition
            for cell_id in sig_cells[:1]:  # Plot first 3 cells of each type (:3)
                # for trials, comb, label in all_conditions:
                # Create valid filename by replacing problematic characters
                # safe_label = label.replace('/', '_').replace(' ', '_')
                
                cell_viz.plot_informative_cell_overlay_minimal_axis(
                    aligned_imaging=data['aligned_imaging'],
                    cell_id=cell_id,
                    all_conditions=data['all_conditions'], #condition_indices=trials,
                    condition_colors=condition_colors,
                    title_base=f'{animalID} {date} {cell_type} Cell {cell_id} ', #- {label}
                    peak_info = peaks_by_celltype[dataset][cell_type]['sc']['sc_instantaneous_information_mean']['peak_values'][cell_id],
                    subplot_split='right',
                    orientation='vertical', figsize=(2,4),
                    smoothing=None,
                    shading = False,
                    save_path='' #f'{base_figure_dir}/example_cells/{animalID}_{date}_{cell_type}_cell_{cell_id}' #_{safe_label}
                )

In [ ]:
# Plot special neurons for each dataset
to_decode = 'sound_category'
base_figure_dir = f'W:\Connie/results\Bassi2025/fig2\glm_decoding\{to_decode}' #f'V:\Connie/results\glm_decoding/2025/figures/all_balanced\{to_decode}'
os.makedirs(base_figure_dir+'/example_cells', exist_ok=True)
condition_colors = ['indigo','mediumpurple','teal','turquoise']#['black','gray','black','gray'] #['navy','royalBlue','darkred','firebrick'] #['#377eb8', '#e41a1c']  # blue and red

#get the peaks by celltype
current_model_type = 'pre' #'pre' or passive
results_pre_all = results_pre_all_sb #results_to_test #results_pre_all_sb
start_frame = 14 #14 before this there is artifact
end_frame = 100 #100 for sound/photostim
threshold = 0.06 #shuffled_threshold['universal_threshold'] #np.max(list(shuffled_threshold['thresholds'].values())) #shuffled_threshold['universal_threshold'] #0.06 #shuffled_threshold['universal_threshold'] #use threshold OR percentile to decide signficant neurons
method = 'threshold_peak' #'threshold_peak'# 'shuffled_peak', 'threshold_peak', 'range_threshold', 'combined'
metric_to_plot = 'sc_instantaneous_information_mean'
current_decoder_type = to_decode #shuffled/

# Analyze peaks by cell type
peaks_by_celltype = []
peaks_by_celltype = analysis_dec.analyze_peaks_by_celltype( results_pre_all,shuffled_structure, method = method,
                                                            decoder_type=current_decoder_type, start_frame=start_frame , end_frame = end_frame,
                                                            significance_percentile = prtile, threshold = threshold, window = 3) #, end_frame = 100

print(f"Peaks by cell type: {peaks_by_celltype.keys()}")
for dataset in peaks_by_celltype.keys(): #sig_neurons: #specified_datasets:
    # Extract animal ID and date from dataset name
    animalID, date = dataset.split('_')
    data = aligned_data[(animalID, date)]
    
    # Get significant neurons and their modulation values
    sig_neurons = None #significant_neurons_mod[dataset][0]
    mod_values = None #mod_index_neurons[dataset]
    
    # Find special neurons
    special_neurons = analysis_dec.find_special_neurons(dataset, None, None, peaks_by_celltype, cell_type_filter='som')

    
    for i, (category, info) in enumerate(special_neurons.items()):
        if info['neuron_id'] is not None:
            cur_celltype= info['cell_type']
            cur_cel_id = info['neuron_id']

            cell_viz.plot_avg_informative_cell_overlay(
                    results=results_pre_all_sb[dataset],
                    decoded_variable=to_decode,
                    aligned_imaging=data['aligned_imaging'],
                    cell_id=info['neuron_id'],
                    all_conditions=data['all_conditions'], #condition_indices=trials,
                    condition_colors=condition_colors,
                    title_base=f'{animalID} {date} {cur_celltype} Cell {cur_cel_id}', #- {label}
                    peak_info = info['info_val'], #peaks_by_celltype[dataset][cell_type]['sc']['sc_instantaneous_information_mean']['peak_values'][cell_id],
                    subplot_split='Right',
                    orientation='vertical', figsize=(1.5,1.5),
                    smoothing=None,
                    shading = False,
                    save_path=f'{base_figure_dir}/example_cells/updated_info_deconv_{animalID}_{date}_{cur_celltype}_cell_{cur_cel_id}.pdf', #_{safe_label},
                    plot_information = True,  # NEW: whether to plot peak info line
                    combine_groups = True,  # NEW: whether to combine groups in one plot
                    linewidth = 0.5
                )
            
            #this plots original plots separated by conditions
            # cell_viz.plot_informative_cell(
            #     aligned_imaging=data['aligned_imaging'],
            #     cell_id=info['neuron_id'],
            #     all_conditions=data['all_conditions'],
            #     title_base=f'{category}\nMI: {info["mod_val"]:.3f}, Info: {info["info_val"]:.3f}\n{info["cell_type"]} Cell {cur_cel_id }',
            # )
            
    
    # plt.tight_layout()
    # plt.savefig(f'{base_figure_dir}/{animalID}_{date}_special_neurons.pdf', 
    #             bbox_inches='tight', dpi=300)
    # plt.close()

    

print("Special neurons plotting complete!")

In [ ]:
#LOAD SIGNIFICANTLY MODULATED CELLS (FROM MATLAB)
import scipy

opto = 0
opto_dir = 'V:/Connie/results/opto_2024/context/mod' #'V:/Connie/results/passive/mod' #'V:/Connie/results/opto_2024/context/mod' #'V:/Connie/results/active/mod' #


# Load the condition_array_trials structure
mat_data = scipy.io.loadmat(os.path.join(opto_dir,'info.mat'))
info = mat_data['info'][0][0]

# Assuming your mouse_date structure is loaded as a numpy array
mouse_dates = [
    item[0].replace('\\', '_').replace('/', '_')  # Replace slashes with underscores for consistency
    for item in info['mouse_date'][0]
]
print(mouse_dates)


current_context = 0
if current_context == 2:
    mod_idx_contexts = scipy.io.loadmat('V:\Connie/results\opto_sound_2025\context\mod\prepost\separate\mod_indexm.mat')
else:
    mod_idx_contexts = scipy.io.loadmat('V:\Connie/results\opto_sound_2025\context\mod\ctrl\separate\mod_indexm.mat')
mod_index_contexts = mod_idx_contexts['mod_indexm']

sig_cels = scipy.io.loadmat('V:\Connie/results\opto_sound_2025\context\mod\prepost\opto_sig_cells.mat') #SOUND OPTO CELLS scipy.io.loadmat('V:\Connie/results\opto_sound_2025\context\sounds\mod\prepost_sound\separate\sound_sig_cells.mat')
sig_cells = sig_cels['opto_sig_cells']-1 #minus one bc of MATLAB indexing


# Create an empty dictionary to store the significant neurons for each mouse_date
significant_neurons_mod = {}
mod_index_neurons = {}
mod_index_neurons_all = {}



# Iterate over the mouse_dates and map to corresponding neurons in sig_cells by index
for idx, mouse_date in enumerate(mouse_dates):
    if idx < len(sig_cells):
        # Get the significant neurons for the current index
        significant_neurons_al = sig_cells[idx][0]
        print(f"Mouse date: {mouse_date}, Significant neurons: {significant_neurons_al}")   
        # Store the results in a dictionary with mouse_date as the key
        significant_neurons_mod[mouse_date] = significant_neurons_al
        current_mod = mod_index_contexts[idx,current_context][0] #mod_index[idx][0][0] #[dataset,context][0]
        # print(f"Mouse date: {mouse_date}, Modulation index: {current_mod}")
        mod_index_neurons[mouse_date] = current_mod[significant_neurons_al]
        mod_index_neurons_all[mouse_date] = current_mod
    else:
        print(f"No significant neurons found for {mouse_date}")

# Now you have a dictionary mapping each dataset to its significant neurons

print(f'total significant neurons: {len(sig_cells)}')

In [ ]:
#PLOT MODULATION INDEX VS PEAK INFORMATION


# Create figure for scatter plot
plt.figure(figsize=(8, 6))

# Store all values for correlation analysis
all_peaks = []
all_mods = []


#recalculate peaks by cell type for this context
if current_context == 0:
    results_pre_all =results_pre_all_sb
    shuffled_to_use = shuffled_structure
else:
    results_pre_all = results_pass_all_sb 
    shuffled_to_use = shuffled_structure_pass

start_frame = 14 #14 before this there is artifact
end_frame = 100 #100 for sound/photostim
threshold =  0.06 #use threshold OR percentile to decide signficant neurons
method = 'threshold_peak'# 'shuffled_peak', 'threshold_peak', 'range_threshold', 'combined'
metric_to_plot = 'sc_instantaneous_information_mean'
current_decoder_type = 'sound_category' #shuffled/
prtile = 90 

# Analyze peaks by cell type
peaks_by_celltype = []
peaks_by_celltype = analysis_dec.analyze_peaks_by_celltype( results_pre_all, shuffled_to_use, method = method,
                                                            decoder_type=current_decoder_type, start_frame=start_frame , end_frame = end_frame,
                                                            significance_percentile = prtile, threshold = threshold, window = 3)

# Get list of datasets except the last one
specified_datasets = list(peaks_by_celltype.keys())#list(significant_neurons_mod.keys())[:-1]

# Iterate through each dataset
for dataset in specified_datasets:
    # Get significant neurons for this dataset
    sig_neurons = significant_neurons_mod[dataset]
    mod_values = np.abs(mod_index_neurons[dataset])
    
    # For each cell type, find matching neurons and their peak values
    for cell_type in ['pyr', 'pv', 'som']:
        if cell_type in peaks_by_celltype[dataset]:
            # Get global indices and peak values for this cell type
            global_indices = peaks_by_celltype[dataset][cell_type]['sc']['sc_instantaneous_information_mean']['global_indices']
            peak_values = peaks_by_celltype[dataset][cell_type]['sc']['sc_instantaneous_information_mean']['peak_values']
            
            # Find which significant neurons belong to this cell type
            matching_neurons = np.isin(global_indices, sig_neurons)
            # print(f' {cell_type} {matching_neurons}')
            
            if np.any(matching_neurons):
                # Get the corresponding mod indices
                mod_idx = np.isin(sig_neurons, global_indices)
                # print(f'{cell_type} {mod_idx}')
                
                # Plot scatter for matched neurons
                plt.scatter(mod_values[mod_idx], 
                          peak_values[matching_neurons],
                          alpha=0.6,
                          color=plotter.celltypecolors[cell_type],
                          label=f'{dataset}_{cell_type}')
                
                # Store matched values for correlation
                all_peaks.extend(peak_values[matching_neurons])
                all_mods.extend(mod_values[mod_idx])

# Calculate correlation and add trend line
all_peaks = np.array(all_peaks)
all_mods = np.array(all_mods)

correlation = np.corrcoef(all_mods, all_peaks)[0,1]
pvalue = scipy.stats.pearsonr(all_mods, all_peaks)[1]

# Rest of plotting code remains the same

# Calculate correlation and add trend line
all_peaks = np.array(all_peaks)
all_mods = np.array(all_mods)

# Calculate correlation and p-value
correlation = np.corrcoef(all_mods, all_peaks)[0,1]
pvalue = scipy.stats.pearsonr(all_mods, all_peaks)[1]

# Fit and plot trend line
z = np.polyfit(all_mods, all_peaks, 1)
p = np.poly1d(z)
x_range = np.linspace(min(all_mods), max(all_mods), 100)
plt.plot(x_range, p(x_range), "r--", alpha=0.8)
# Define horizontal_line as the range of x-values

plt.axhline(y=0.06, color='blue', linestyle='--', alpha=0.7, label='Threshold = 0.06')
# Customize plot
plt.xlabel('|Modulation Index|')
plt.ylabel('Peak Information (bits)')
plt.title(f'Peak Information vs Modulation Index\nr = {correlation:.2f}, p = {pvalue:.3e}')
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
#set plot limits
plt.xlim(0, 1)
plt.ylim(0, 0.5)

#print percentage of significant neurons above threshold
# Calculate the percentage of significant neurons above the threshold
threshold = 0.06
percentage_above_threshold = np.sum(all_peaks > threshold) / len(all_peaks) * 100
print(f"Percentage of significant neurons above threshold {threshold}: {percentage_above_threshold:.2f}%")

# # Save figure
# plt.savefig(os.path.join(base_figure_dir, 'peak_info_vs_mod_index.pdf'), 
#             bbox_inches='tight', 
#             dpi=300)
# plt.close()